In [48]:
from transformers import AutoConfig
import __init__
configs = {
    'DeepSeek-R1-Distill-Qwen-1.5B': AutoConfig.from_pretrained('deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'),
    'DeepSeek-R1-Distill-Qwen-32B': AutoConfig.from_pretrained('deepseek-ai/DeepSeek-R1-Distill-Qwen-32B'),
    'DeepSeek-R1-Distill-Llama-8B': AutoConfig.from_pretrained('deepseek-ai/DeepSeek-R1-Distill-Llama-8B'),
    'QwQ-32B': AutoConfig.from_pretrained('Qwen/QwQ-32B'),
    'Phi-4-reasoning-plus': AutoConfig.from_pretrained('microsoft/Phi-4-reasoning-plus'),
    'DeepSeek-R1-Distill-Qwen-7B': AutoConfig.from_pretrained('deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'),
    'Qwen2.5-7B-Instruct': AutoConfig.from_pretrained('Qwen/Qwen2.5-7B-Instruct'),
    'DeepSeek-R1-Distill-Qwen-14B': AutoConfig.from_pretrained('deepseek-ai/DeepSeek-R1-Distill-Qwen-14B'),
}
change_names = {
    'phi14b': 'Phi-4-reasoning-plus',
    'qwq': 'QwQ-32B',
    '1b':'DeepSeek-R1-Distill-Qwen-1.5B',
    '32b':'DeepSeek-R1-Distill-Qwen-32B',
    '8b':'DeepSeek-R1-Distill-Llama-8B',
    '7b':'DeepSeek-R1-Distill-Qwen-7B',
    '14b':'DeepSeek-R1-Distill-Qwen-14B',
    '7b-instruct':'Qwen2.5-7B-Instruct',
}

In [49]:
keywords = {"wait", "hmm", "alternatively", 'double-check'}
import re

def segement(text):
    text = text.strip()  # 处理原始文本

    # **按 `\n\n` 进行初步分割**
    sections = re.split(r'\n\n+', text)

    # **存储最终分割的部分**
    filtered_sections = []
    temp_section = ""

    for section in sections:
        last_20_chars = section[:30].lower()  # 取后 20 个字符并转换为小写
        # print(last_20_chars)
        if any(keyword in last_20_chars for keyword in keywords):
            # 如果当前 `temp_section` 不是空的，则合并存储
            if temp_section:
                filtered_sections.append(temp_section.strip())
            temp_section = section+ "\n\n"
        else:
            # 累积无关键词的部分，等待下次匹配
            temp_section += section + "\n\n"

    # **如果最后 `temp_section` 仍有内容，加入最终结果**
    if temp_section:
        filtered_sections.append(temp_section.strip())
    return len(filtered_sections)

In [50]:
def many_decode_flops(p_l, s, h, h_, n):
    # return 8*s*(h^2) + 16*s*h + 4*s*p_l*h + 4*s*p_l*n + 6*s*h*h_ + 2*s*h_
    # return cum_decode_flop(p_l, s, h, h_, n)/5
    return decode_flop(p_l, h, h_, n)

def prefill_flop(s, h, h_, n):
    return 8*s*(h^2) + 16*s*h + 4*(s^2)*h + 4*(s^2)*n + 6*s*h*h_ + 2*s*h_

def decode_flop(s, h, h_, n):
    return 8*(h^2) + 16*h + 4*s*h + 4*s*n + 6*h*h_ + 2*h_

def cum_decode_flop(p_l, l, h, h_, n):
    flops = 0
    for i in range(l):
        flops = flops + decode_flop(p_l+i, h, h_, n)
    return flops

def get_flops(name, help_model, token_usages):
    correct_num = 0
    prefill_flops, prefix_flops, decode_flops=0,0,0
    if help_model not in configs.keys():
        config = configs[name]
        info= config.hidden_size, config.intermediate_size, config.num_attention_heads
        prefill_flops = prefill_flop(token_usages['prompt_tokens'], *info) 
        decode_flops = cum_decode_flop(token_usages['prompt_tokens'], token_usages['completion_tokens'], *info)
    else:
        spe_config = configs[name]
        tgt_config = configs[help_model]
        spe_info = spe_config.hidden_size, spe_config.intermediate_size, spe_config.num_attention_heads
        tgt_info = tgt_config.hidden_size, tgt_config.intermediate_size, tgt_config.num_attention_heads
        correct_tokens = token_usages['correct_tokens']
        completion_tokens = token_usages['completion_tokens']
        prompt_tokens = token_usages['prompt_tokens']
        last_spe_decode_pos = prompt_tokens
        # correct_times = len(token_usages['correct_tokens'])
        # print(token_usages['correct_tokens'])
        # print(correct_times)
        token_num = 0
        prefill_flops = prefill_flops+prefill_flop(prompt_tokens, *spe_info)
        prefill_flops = prefill_flops+prefill_flop(prompt_tokens, *tgt_info)
        for cor_t in correct_tokens:
            pos, token_num = cor_t['pos']+prompt_tokens, cor_t['token_num']
            decode_flops = decode_flops+cum_decode_flop(last_spe_decode_pos, pos-last_spe_decode_pos, *spe_info)
            f = many_decode_flops(last_spe_decode_pos, pos-last_spe_decode_pos, *tgt_info)
            prefix_flops = prefix_flops+f #*f/cum_decode_flop(last_spe_decode_pos, pos-last_spe_decode_pos, *tgt_info)
            decode_flops = decode_flops+cum_decode_flop(pos, token_num, *tgt_info)
            f = many_decode_flops(pos, token_num, *spe_info)
            prefix_flops = prefix_flops+f #*f/cum_decode_flop(pos, token_num, *spe_info)
            correct_num = correct_num + token_num
            last_spe_decode_pos = pos
        decode_flops = decode_flops+cum_decode_flop(last_spe_decode_pos+token_num, completion_tokens+prompt_tokens-last_spe_decode_pos-token_num, *spe_info)
    return prefill_flops, prefix_flops, decode_flops, correct_num


In [51]:
import os
import json
import math
import re

def log_flops(flops):
    # print(flops)
    if flops <= 0: return 0
    r = 312*10**8
    # print(flops/r)
    return flops/r #

def count_exact_wait_in_code(code_str):
    # count = code_str.lower().count('wait')
    count = segement(code_str.lower())
    # print(count)
    # matches = re.findall(r'\bwait\b', code_str.lower(), flags=re.IGNORECASE)
    # # print(f"'wait' 精确出现次数: {len(matches)}")
    # count = len(matches)
    return count

def get_info(data, name, help_model, t=0.6):
    correct_num, correct_tokens, worng_tokens, all_time_spend = 0, 0, 0, 0
    fix_tokens, prefill_flops, prefix_flops, decode_flops, = 0, 0, 0, 0
    right_wait_count,wrong_wait_count = 0,0
    for k in data.keys():
        response = data[k]['responses'][str(t)][0]['content']
        wait_count = count_exact_wait_in_code(response)
        token_usages = data[k]['token_usages'][str(t)][0]
        # if 'correct_tokens' not in token_usages.keys():
        #     print(token_usages.keys())
        #     print(k)
        #     print(token_usages)
        # time_spend = token_usages['time_spend']
        tokens_num = token_usages['completion_tokens']
        prefill_flop, prefix_flop, decode_flop, fix_token = get_flops(name, help_model, token_usages)
        prefill_flops = prefill_flops + prefill_flop
        prefix_flops = prefix_flops + prefix_flop
        decode_flops = decode_flops + decode_flop
        if data[k]['responses'][str(t)][0]['correctness']: 
            correct_num = correct_num+1
            correct_tokens = correct_tokens+tokens_num
            right_wait_count = right_wait_count + wait_count
        else:
            worng_tokens = worng_tokens+tokens_num
            wrong_wait_count = wrong_wait_count + wait_count
        fix_tokens = fix_tokens+fix_token
        #  all_time_spend = all_time_spend + time_spend
    all_num = len(data.keys())
    return {
        'right_wait_count': right_wait_count,
        'wrong_wait_count': wrong_wait_count,
        'all_num':all_num,
        'correct_num':correct_num,
        'wrong_num':all_num-correct_num,
        'acc':round(correct_num/all_num, 4),
        'avg_token_num': round((correct_tokens+worng_tokens)/all_num, 2),
        'correct_avg_tokens':round(correct_tokens/correct_num, 2) if correct_num != 0 else 0,
        'worng_avg_tokens':round(worng_tokens/(all_num-correct_num), 2) if all_num-correct_num != 0 else 0,
        'prefill_flops': round(log_flops(prefill_flops), 2),
        'prefix_flops': round(log_flops(prefix_flops), 2),
        'decode_flops': round(log_flops(decode_flops), 2),
        'all_tokens': correct_tokens+worng_tokens,
        't/flops': round((correct_tokens+worng_tokens)/log_flops(decode_flops+prefix_flops+prefill_flops), 2),
        'all_flops': round(log_flops(decode_flops+prefix_flops+prefill_flops), 2),
        #  'avg_time_spend': round((correct_tokens+worng_tokens)/all_time_spend, 2),
        'fix_ratio': round(fix_tokens/(correct_tokens+worng_tokens),2),
        'fix_tokens':fix_tokens
    }

## normal

In [52]:
from collections import defaultdict
results = defaultdict(dict)

aime_data = dict()
folder_path = "/home/wxy320/ondemand/program/speculative_thinking/eval_all_results/rebuttal/normal"  # 替换为你的文件夹路径
json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]
for file in json_files:
    name = file.split('_')[0]
    dataset = file.split('_')[1]
    file_path = os.path.join(folder_path, file)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        result = get_info(data, name, None)
    if 'aime' in dataset:
        if name not in aime_data.keys():
            aime_data[name] = dict()
        aime_data[name].update(data)
    results[(name,dataset)] = {
        'normal': result
    }
for name, data in aime_data.items():
    result = get_info(data, name, None)
    results[(name,'aime')] = {
        'normal': result
    }

In [53]:
final_results = defaultdict(dict)
for key, r in results.items():
    # print(key)
    final_results[key[1]][key[0]]=r
for d, r in final_results.items():
    if d == 'math500': 
        print(d)
        print('------------------------')
        for k, r_ in r.items():
            print(k)
            for k_, r_1 in r_.items():
                print(k_, r_1['acc'], r_1['all_tokens'], r_1['t/flops'])
        print('------------------------')

math500
------------------------
Phi-4-reasoning-plus
normal 0.934 3436848 37.61
DeepSeek-R1-Distill-Llama-8B
normal 0.858 2256727 69.66
DeepSeek-R1-Distill-Qwen-1.5B
normal 0.832 2719563 242.56
DeepSeek-R1-Distill-Qwen-32B
normal 0.928 1901073 32.21
QwQ-32B
normal 0.95 2198515 32.38
------------------------


In [54]:
folder_path = "/home/wxy320/ondemand/program/speculative_thinking/eval_all_results/rebuttal/speculative"  # 替换为你的文件夹路径
json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]

aime_data = dict()
for file in json_files:
    name, model, dataset = change_names[file.split('_')[0]], change_names[file.split('_')[1].split('.')[0]], file.split('_')[2]
    file_path = os.path.join(folder_path, file)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        result = get_info(data, name, model)
        # print(result)
    if 'aime' in dataset:
        # print(name, model, dataset)
        if (name, model) not in aime_data.keys():
            aime_data[(name, model)] = dict()
        aime_data[(name, model)].update(data)
    results[(name,dataset)][model] = result
for (name, model), data in aime_data.items():
    # print(name, model, len(data))
    result = get_info(data, name, model)
    results[(name,'aime')][model] = result

In [55]:
final_results = defaultdict(dict)
for key, r in results.items():
    # print(key)
    final_results[key[1]][key[0]]=r
for d, r in final_results.items():
    if d == 'math500' or d == 'aime': 
        print(d)
        print('------------------------')
        for k, r_ in r.items():
            print(k)
            for k_, r_1 in r_.items():
                # print(r_1)
                print('+', k_, r_1['acc'], r_1['avg_token_num'], r_1['t/flops'], r_1['fix_ratio'])
        print('------------------------')

math500
------------------------
Phi-4-reasoning-plus
+ normal 0.934 6873.7 37.61 0.0
DeepSeek-R1-Distill-Llama-8B
+ normal 0.858 4513.45 69.66 0.0
+ DeepSeek-R1-Distill-Qwen-32B 0.892 4082.24 46.49 0.21
DeepSeek-R1-Distill-Qwen-1.5B
+ normal 0.832 5439.13 242.56 0.0
+ Phi-4-reasoning-plus 0.9 3768.93 97.95 0.2
+ QwQ-32B 0.91 5483.38 86.43 0.22
DeepSeek-R1-Distill-Qwen-32B
+ normal 0.928 3802.15 32.21 0.0
QwQ-32B
+ normal 0.95 4397.03 32.38 0.0
------------------------
aime
------------------------
Phi-4-reasoning-plus
+ normal 0.7222 14230.63 39.11 0.0
DeepSeek-R1-Distill-Llama-8B
+ normal 0.3667 14869.61 60.31 0.0
+ DeepSeek-R1-Distill-Qwen-32B 0.5222 13036.79 41.63 0.22
DeepSeek-R1-Distill-Qwen-1.5B
+ normal 0.2556 17798.4 198.85 0.0
+ Phi-4-reasoning-plus 0.3778 16438.44 103.46 0.16
+ QwQ-32B 0.4222 20428.94 79.27 0.2
DeepSeek-R1-Distill-Qwen-32B
+ normal 0.6556 12274.31 29.97 0.0
QwQ-32B
+ normal 0.7778 13977.46 29.75 0.0
------------------------


In [15]:
from collections import defaultdict
results = defaultdict(dict)

folder_path = "/home/wxy320/ondemand/program/speculative_thinking/eval_all_results/rebuttal/ablation" 
json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]

aime_data = dict()
for file in json_files:
    name, model, dataset = change_names[file.split('_')[0]], change_names[file.split('_')[1].split('.')[0]], file.split('_')[3]
    sit = file.split('_')[2]
    file_path = os.path.join(folder_path, file)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        result = get_info(data, name, model)
        # print(result)
    if 'aime' in dataset:
        # print(name, model, dataset)
        if (name, model) not in aime_data.keys():
            aime_data[(name, model, sit)] = dict()
        aime_data[(name, model, sit)].update(data)
    results[(name,dataset,sit)][model] = result
for (name, model, sit), data in aime_data.items():
    # print(name, model, len(data))
    result = get_info(data, name, model)
    results[(name,'aime',sit)][model] = result

In [44]:
final_results = defaultdict(dict)
for key, r in results.items():
    # print(key)
    final_results[key[1]][key[0]+'_'+key[2]]=r
for d, r in final_results.items():
    if d == 'math500' or d == 'aime': 
        print(d)
        print('------------------------')
        for k, r_ in r.items():
            print(k)
            for k_, r_1 in r_.items():
                # print(r_1)
                print('+', k_, r_1['acc'], r_1['avg_token_num'], r_1['t/flops'])
        print('------------------------')

IndexError: tuple index out of range

In [19]:
from collections import defaultdict
results = defaultdict(dict)

folder_path = "/home/wxy320/ondemand/program/speculative_thinking/eval_all_results/rebuttal/inital" 
json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]

aime_data = dict()
for file in json_files:
    name, model, dataset = change_names[file.split('_')[0]], change_names[file.split('_')[1].split('.')[0]], file.split('_')[3]
    sit = file.split('_')[2]
    file_path = os.path.join(folder_path, file)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        result = get_info(data, name, model)
        # print(result)
    if 'aime' in dataset:
        # print(name, model, dataset)
        if (name, model) not in aime_data.keys():
            aime_data[(name, model, sit)] = dict()
        aime_data[(name, model, sit)].update(data)
    results[(name,dataset,sit)][model] = result
for (name, model, sit), data in aime_data.items():
    # print(name, model, len(data))
    result = get_info(data, name, model)
    results[(name,'aime',sit)][model] = result

In [20]:
final_results = defaultdict(dict)
for key, r in results.items():
    # print(key)
    final_results[key[1]][key[0]+'_'+key[2]]=r
for d, r in final_results.items():
    if d == 'math500' or d == 'aime': 
        print(d)
        print('------------------------')
        for k, r_ in r.items():
            print(k)
            for k_, r_1 in r_.items():
                # print(r_1)
                print('+', k_, r_1['acc'], r_1['avg_token_num'], r_1['t/flops'])
        print('------------------------')

math500
------------------------
Qwen2.5-7B-Instruct_without.yml
+ DeepSeek-R1-Distill-Qwen-7B 0.738 1009.51 46.55
+ DeepSeek-R1-Distill-Qwen-32B 0.742 1071.67 35.85
------------------------
aime
------------------------
Qwen2.5-7B-Instruct_without.yml
+ DeepSeek-R1-Distill-Qwen-7B 0.0667 1657.17 51.05
+ DeepSeek-R1-Distill-Qwen-32B 0.0667 5347.6 42.26
------------------------


In [56]:
from collections import defaultdict
results = defaultdict(dict)

folder_path = "/home/wxy320/ondemand/program/speculative_thinking/eval_all_results/rebuttal/best"  # 替换为你的文件夹路径
json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]

aime_data = dict()
for file in json_files:
    name, model, dataset = change_names[file.split('_')[0]], change_names[file.split('_')[1].split('.')[0]], file.split('_')[2]
    file_path = os.path.join(folder_path, file)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        result = get_info(data, name, model)
        # print(result)
    if 'aime' in dataset:
        # print(name, model, dataset)
        if (name, model) not in aime_data.keys():
            aime_data[(name, model)] = dict()
        aime_data[(name, model)].update(data)
    results[(name,dataset)][model] = result
for (name, model), data in aime_data.items():
    # print(name, model, len(data))
    result = get_info(data, name, model)
    results[(name,'aime')][model] = result

In [57]:
final_results = defaultdict(dict)
for key, r in results.items():
    # print(key)
    final_results[key[1]][key[0]]=r
for d, r in final_results.items():
    if d == 'math500' or d == 'gpqa':
        print(d)
        print('------------------------')
        for k, r_ in r.items():
            print(k)
            for k_, r_1 in r_.items():
                # print(r_1)
                print('+', k_, r_1['acc'], r_1['avg_token_num'], r_1['t/flops'], r_1['fix_ratio'])
        print('------------------------')

gpqa
------------------------
DeepSeek-R1-Distill-Qwen-1.5B
+ DeepSeek-R1-Distill-Qwen-7B 0.3889 7467.97 126.94 0.17
+ DeepSeek-R1-Distill-Qwen-14B 0.3889 8134.34 128.13 0.15
+ DeepSeek-R1-Distill-Qwen-32B 0.4192 7612.36 91.78 0.17
------------------------
math500
------------------------
DeepSeek-R1-Distill-Qwen-1.5B
+ DeepSeek-R1-Distill-Qwen-7B 0.858 5158.69 135.88 0.18
+ DeepSeek-R1-Distill-Qwen-14B 0.89 4527.35 134.64 0.19
+ DeepSeek-R1-Distill-Qwen-32B 0.894 4582.78 96.61 0.19
------------------------
